# The Vendor Fraud Review Op as an executable specification

The whitepaper's running example — the **Vendor Fraud Review Op** and its
Validation Strategy manifest (*The Distributed AI Economy*, Revision 9,
§5.6) — implemented on an RDF + SysML substrate with SKOS, PROV-O, EARL,
SHACL, and OpenSysML.

**Everything below is executed.** Every claim on this page is the output of
the code cell above it, produced live against the pinned source and the
committed files. Nothing is asserted by prose alone.

## 1 · Toolchain

The model runs on a pinned OpenSysML binary, fetched and verified against the
release's SHA256SUMS.

In [ ]:
import subprocess, pathlib
ROOT = pathlib.Path.cwd()
print(subprocess.run(["bash", "toolchain/get-sysml.sh"], capture_output=True, text=True).stdout or "already installed")
print(subprocess.run(["toolchain/bin/sysml", "--version"], capture_output=True, text=True).stdout.strip())

## 2 · The pinned source

Every definition and every gate below traces to one content-addressed snapshot
of the whitepaper. The hash the RDF declares must be the hash the file has.

In [ ]:
import hashlib, rdflib
pdf = next((ROOT / "sources").glob("*.pdf"))
actual = hashlib.sha256(pdf.read_bytes()).hexdigest()
g = rdflib.Graph()
g.parse("sources/sources.ttl"); g.parse("vocabulary/op-concepts.ttl")
VFR = "https://example.org/vfr#"
declared = str(next(g.objects(None, rdflib.URIRef(VFR + "contentHash"))))
print(f"snapshot file : {pdf.name}")
print(f"sha256(file)  : {actual}")
print(f"declared hash : {declared}")
print(f"match         : {actual == declared}")

## 3 · Vocabulary — his definitions, verbatim

Ten concepts, each with a `skos:definition` lifted exactly from the paper and
a section locator. The cell checks each definition against the PDF text
itself: an invented or paraphrased definition would print False.

In [ ]:
import re
from rdflib.namespace import SKOS
norm = lambda t: re.sub(r"\s+", "", re.sub(r"\([^)]*\)", "", t)).lower()
pdf_text = norm(subprocess.run(["pdftotext", "-layout", str(pdf), "-"], capture_output=True, text=True).stdout)
rows = []
for c in g.subjects(SKOS.definition, None):
    label = str(next(g.objects(c, SKOS.prefLabel)))
    definition = str(next(g.objects(c, SKOS.definition)))
    locator = str(next(g.objects(c, rdflib.URIRef(VFR + "locator"))))
    rows.append((locator, label, norm(definition) in pdf_text, definition))
for locator, label, verbatim, definition in sorted(rows):
    print(f"{locator:6} {label:24} verbatim: {verbatim}")
    print(f"       {definition[:96]}...")

## 4 · The Op as an assembly — gates, seams, and aggregates

The §5.6 gates block, verbatim:

```yaml
gates:
  - if: confidence < 0.80
    then: human_review_required
  - if: consensus_disagreement > 0.25
    then: expert_review_required
  - if: sensitive_data_detected
    then: stop_and_escalate
  - if: vendor_risk == high
    then: human_approval_required
```

The Op is modeled as an **assembly of abstract and concrete parts wired
along ports**: four oracle components feed a policy engine over reading
seams; raised obligations cross to the human actions; clearance crosses to
the anomaly-summary cog, whose output the assembly's boundary exposes. One
toolchain then checks three levels at once:

- **component** — each `if/then` line as an `implies` constraint on the
  policy engine (the verbatim gates, with the numerical parameters factored
  as single-point definitions);
- **wiring** — seam integrity: the value the engine ruled on is the value
  the oracle returned (WIRE-01..04), plus structural seam conformance in
  section 7;
- **system** — properties only the composed Op has: every raised obligation
  discharged across components (SYSTEM-01), a stopped Op emits nothing
  (SYSTEM-02, §5.2 verbatim: "the Op stops before external transmission"),
  and the aggregate outcome — **executed vs noOp** — coheres with what was
  emitted (SYSTEM-03). Navigating the rules and executing are different
  facts, and the model now says which happened.

Adjudicated interpretations are logged in `open-questions/`; the drawn
topology itself is claimed as this formalization's contribution (GAP-08) —
an explicit, checkable dataflow is what the computational representation
adds that the prose could not carry. The satisfy sweep evaluates all three
levels together.

In [ ]:
model = "model/vendor-fraud-review.sysml"
source = pathlib.Path(model).read_text()
params_block = re.search(r"part def ValidationStrategyParameters .*?\n        }", source, re.S).group(0)
print(params_block, "\n")
for gate_block in re.findall(r"requirement def <'GATE-\d+'>.*?\n        }", source, re.S)[:1]:
    print(gate_block, "\n")
print("\n".join(l.strip() for l in source.splitlines() if "connect" in l and "interface" in l), "\n")
r = subprocess.run(["toolchain/bin/sysml", model, "-validate", "-strict"], capture_output=True, text=True)
print(f"validate -strict exit code: {r.returncode}")
r = subprocess.run(["toolchain/bin/sysml", model, "-satisfy=RunConfigurations"], capture_output=True, text=True)
print(r.stdout.strip())
print(f"satisfy exit code: {r.returncode}")

The specification can also say **no** — once per check level. The
counterexample file holds three runs: the gate fired but no obligation was
raised (component), the obligation was raised and never discharged
(system), and the Op stopped but transmitted anyway (aggregate). The same
algebra fails each, naming the violated implication:

In [ ]:
r = subprocess.run(["toolchain/bin/sysml", "counterexamples/run-unattended.sysml", "-satisfy=RunConfigurations"], capture_output=True, text=True)
print(r.stdout.strip())
print(f"satisfy exit code: {r.returncode}")

## 5 · Worked examples — the policies invoked, gate by gate

Narrated scenarios of the Op at work. Every fact the policy reads comes from
an oracle (section 11's interface contract), so the scenarios **mock the
oracles** and drive the mocked facts through the specification itself: each
run below reuses the committed policy text byte-identical, binds a scenario
into the assembly's components by nested redefinition, and asks the pinned
evaluator for the answer — component, wiring, and system checks together.
Every scenario ends by naming its **aggregate**: did the Op execute, or was
it a deliberate no-op?

In [ ]:
import tempfile, json
SPEC_SOURCE = pathlib.Path(model).read_text()
POLICY_TEXT = SPEC_SOURCE.split("    package RunConfigurations {")[0]
print(f"policy reused byte-identical from {model} "
      f"(sha256 {hashlib.sha256(POLICY_TEXT.encode()).hexdigest()[:12]})")

SERVICES = {
    "confidence": "extraction-confidence service",
    "consensus_disagreement": "consensus-comparator service",
    "sensitive_data_detected": "sensitive-data-scanner service",
    "vendor_risk": "vendor-risk-cog scoring endpoint",
}

def mock_oracle(variable, value):
    payload = json.dumps({"op": "vendor-fraud-review", "variable": variable})
    print(f"  mock oracle {SERVICES[variable]}: sent {payload} -> 200 {json.dumps({variable: value})}")
    return value

def b(v):
    return "true" if v else "false"

SCENARIO_TEMPLATE = """    package RunConfigurations {
        part def Scenario :> Manifest::VendorFraudReviewOp {
            part :>> confidenceOracle { attribute :>> returnedConfidence = @CONF@; }
            part :>> consensusOracle { attribute :>> returnedDisagreement = @CONS@; }
            part :>> scannerOracle { attribute :>> returnedDetected = @SENS@; }
            part :>> riskOracle { attribute :>> returnedRisk = Manifest::RiskLevel::@RISK@; }
            part :>> policyEngine {
                attribute :>> confidence = @CONF@;
                attribute :>> confidenceLevel = Manifest::ConfidenceLevel::@LEVEL@;
                attribute :>> consensusDisagreement = @CONS@;
                attribute :>> sensitiveDataDetected = @SENS@;
                attribute :>> vendorRisk = Manifest::RiskLevel::@RISK@;
                attribute :>> humanReviewRequired = @HRR@;
                attribute :>> expertReviewRequired = @ERR@;
                attribute :>> stopAndEscalate = @SAE@;
                attribute :>> humanApprovalRequired = @HAR@;
            }
            part :>> humanAction {
                attribute :>> humanReviewPerformed = @HRP@;
                attribute :>> expertReviewPerformed = @ERP@;
                attribute :>> escalationPerformed = @ESP@;
                attribute :>> humanApprovalGiven = @HAG@;
            }
            part :>> summaryCog { attribute :>> outputEmitted = @OUT@; }
            attribute :>> aggregateOutcome = Manifest::AggregateOutcome::@AGG@;
        }
        part s : Scenario;
        requirement i1 : Manifest::ConfidenceInterface { subject pe = s.policyEngine; }
        requirement g1 : Manifest::ConfidenceGate { subject pe = s.policyEngine; }
        requirement g2 : Manifest::ConsensusGate { subject pe = s.policyEngine; }
        requirement g3 : Manifest::SensitiveDataGate { subject pe = s.policyEngine; }
        requirement g4 : Manifest::VendorRiskGate { subject pe = s.policyEngine; }
        requirement w1 : Manifest::ConfidenceSeamIntegrity { subject op = s; }
        requirement w2 : Manifest::ConsensusSeamIntegrity { subject op = s; }
        requirement w3 : Manifest::ScannerSeamIntegrity { subject op = s; }
        requirement w4 : Manifest::RiskSeamIntegrity { subject op = s; }
        requirement s1 : Manifest::ObligationsDischarged { subject op = s; }
        requirement s2 : Manifest::StopMeansNoOutput { subject op = s; }
        requirement s3 : Manifest::OutcomeCoherence { subject op = s; }
        verification def ScenarioCase {
            objective {
                verify i1; verify g1; verify g2; verify g3; verify g4;
                verify w1; verify w2; verify w3; verify w4;
                verify s1; verify s2; verify s3;
            }
        }
        verification scenarioCase : ScenarioCase;
    }
}
"""

def run_scenario(name, readings, recorded):
    print(f"scenario '{name}'")
    conf = mock_oracle("confidence", readings["confidence"])
    cons = mock_oracle("consensus_disagreement", readings["consensus_disagreement"])
    sens = mock_oracle("sensitive_data_detected", readings["sensitive_data_detected"])
    risk = mock_oracle("vendor_risk", readings["vendor_risk"])
    agg = recorded.get("aggregate", "executed")
    fills = {
        "@CONF@": repr(conf), "@CONS@": repr(cons), "@SENS@": b(sens), "@RISK@": risk,
        "@LEVEL@": recorded["confidenceLevel"],
        "@HRR@": b(recorded.get("humanReviewRequired", False)),
        "@ERR@": b(recorded.get("expertReviewRequired", False)),
        "@SAE@": b(recorded.get("stopAndEscalate", False)),
        "@HAR@": b(recorded.get("humanApprovalRequired", False)),
        "@HRP@": b(recorded.get("humanReviewPerformed", False)),
        "@ERP@": b(recorded.get("expertReviewPerformed", False)),
        "@ESP@": b(recorded.get("escalationPerformed", False)),
        "@HAG@": b(recorded.get("humanApprovalGiven", False)),
        "@OUT@": b(recorded.get("outputEmitted", False)),
        "@AGG@": agg,
    }
    scenario = SCENARIO_TEMPLATE
    for k, v in fills.items():
        scenario = scenario.replace(k, v)
    with tempfile.NamedTemporaryFile("w", suffix=".sysml", delete=False) as f:
        f.write(POLICY_TEXT + scenario)
        path = f.name
    r = subprocess.run(["toolchain/bin/sysml", path, "-satisfy=RunConfigurations"],
                       capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if "satisfy" in line or "Required condition" in line:
            print("  " + line.strip())
    if r.returncode == 0:
        print(f"scenario '{name}': POLICY SATISFIED — aggregate: {agg} (satisfy exit 0)")
    else:
        print(f"scenario '{name}': POLICY VIOLATED (satisfy exit {r.returncode})")

### 5.1 · The happy path

A batch of 37 invoices from a routine vendor. Extraction is clean (0.92),
the three Cogs agree, the scanner finds nothing sensitive, and the risk score
comes back low. **No gate condition holds, so no obligation is raised** — the
Op runs to completion, emits its summary, and the aggregate says so:
executed. This is §4.5's promise ("Close the books") with the governance
machinery present but dormant.

In [ ]:
run_scenario("happy-path",
    readings={"confidence": 0.92, "consensus_disagreement": 0.10,
              "sensitive_data_detected": False, "vendor_risk": "low"},
    recorded={"confidenceLevel": "high", "outputEmitted": True, "aggregate": "executed"})

### 5.2 · GATE-01 binds — and the obligation is discharged

> `- if: confidence < 0.80` / `then: human_review_required` (§5.6, verbatim)

A poor scan drops extraction confidence to 0.71, below the factored
`confidenceThreshold` of 0.80. The gate binds: the obligation
`human_review_required` is raised in the policy engine. The analyst reviews
the low-confidence extractions — the concrete action, in the human-action
component, that discharges the obligation (SYSTEM-01 checks the discharge
ACROSS the two components). The run then executes: output emitted,
aggregate executed. The policy is satisfied **because** the human step
happened.

In [ ]:
run_scenario("low-confidence-reviewed",
    readings={"confidence": 0.71, "consensus_disagreement": 0.10,
              "sensitive_data_detected": False, "vendor_risk": "low"},
    recorded={"confidenceLevel": "low",
              "humanReviewRequired": True, "humanReviewPerformed": True,
              "outputEmitted": True, "aggregate": "executed"})

### 5.3 · GATE-01 binds — and the obligation is neglected

Same readings, but the Op barrels on: the obligation is raised and **no one
reviews**. The specification refuses the run — SYSTEM-01 names the exact
implication that failed. Neglect is not a gap in a workflow diagram; it is
exit 1.

In [ ]:
run_scenario("low-confidence-neglected",
    readings={"confidence": 0.71, "consensus_disagreement": 0.10,
              "sensitive_data_detected": False, "vendor_risk": "low"},
    recorded={"confidenceLevel": "low",
              "humanReviewRequired": True, "humanReviewPerformed": False,
              "outputEmitted": True, "aggregate": "executed"})

### 5.4 · GATE-02 binds — the Cogs disagree

> `- if: consensus_disagreement > 0.25` / `then: expert_review_required` (§5.6, verbatim)

The extraction Cog and the anomaly Cog classify a cluster of invoices
differently; the consensus comparator reports disagreement 0.40, over the
0.25 threshold. Expert review is required — and performed — and the run
executes. (What the disagreement metric actually measures is GAP-04, still
PENDING; the policy exercises only the threshold comparison.)

In [ ]:
run_scenario("split-cogs-expert-reviewed",
    readings={"confidence": 0.92, "consensus_disagreement": 0.40,
              "sensitive_data_detected": False, "vendor_risk": "low"},
    recorded={"confidenceLevel": "high",
              "expertReviewRequired": True, "expertReviewPerformed": True,
              "outputEmitted": True, "aggregate": "executed"})

### 5.5 · GATE-03 binds — sensitive data found, and the Op halts

> `- if: sensitive_data_detected` / `then: stop_and_escalate` (§5.6, verbatim)

The scanner finds an employee's bank details pasted into an invoice memo
field. The stop obligation is raised and discharged (the escalation is
performed) — and, per §5.2, "the Op stops before external transmission":
**no output is emitted, and the aggregate is noOp**. The rules were
navigated successfully AND the aggregate is a deliberate no-op. Those are
two different facts, and the run now states both.

In [ ]:
run_scenario("pii-stop-and-escalate",
    readings={"confidence": 0.92, "consensus_disagreement": 0.10,
              "sensitive_data_detected": True, "vendor_risk": "low"},
    recorded={"confidenceLevel": "high",
              "stopAndEscalate": True, "escalationPerformed": True,
              "outputEmitted": False, "aggregate": "noOp"})

### 5.6 · GATE-03 binds — and the Op transmits anyway

Same escalation, same discharge — but this time the summary goes out the
door and the run calls itself executed. Every gate holds; every obligation
is discharged; and the run is still refused, by the SYSTEM checks:
navigating the rules and then executing regardless of the stop is exactly
the violation a flat rule-list cannot see.

In [ ]:
run_scenario("pii-stopped-but-emitted",
    readings={"confidence": 0.92, "consensus_disagreement": 0.10,
              "sensitive_data_detected": True, "vendor_risk": "low"},
    recorded={"confidenceLevel": "high",
              "stopAndEscalate": True, "escalationPerformed": True,
              "outputEmitted": True, "aggregate": "executed"})

### 5.7 · GATE-04 binds — a high-risk vendor

> `- if: vendor_risk == high` / `then: human_approval_required` (§5.6, verbatim)

The vendor-risk endpoint scores vendor V-2214 `high` (the adjudicated
RiskLevel enumeration, GAP-01 — the trigger value is itself a factored
parameter). No vendor is flagged without a named human's approval; here the
approval is given, and the run executes.

In [ ]:
run_scenario("high-risk-vendor-approved",
    readings={"confidence": 0.92, "consensus_disagreement": 0.10,
              "sensitive_data_detected": False, "vendor_risk": "high"},
    recorded={"confidenceLevel": "high",
              "humanApprovalRequired": True, "humanApprovalGiven": True,
              "outputEmitted": True, "aggregate": "executed"})

### 5.8 · INTERFACE-01 binds — a mislabeled category

The oracle reports 0.71, but the run records its confidence level as `high`.
The human review even happens — yet the documented threshold rule
(INTERFACE-01, the boundary between the oracle's numeric and the category
interface) catches the mislabel and refuses the run. The category layer is
not decoration: it is checked against the number it summarizes.

In [ ]:
run_scenario("mislabeled-confidence",
    readings={"confidence": 0.71, "consensus_disagreement": 0.10,
              "sensitive_data_detected": False, "vendor_risk": "low"},
    recorded={"confidenceLevel": "high",
              "humanReviewRequired": True, "humanReviewPerformed": True,
              "outputEmitted": True, "aggregate": "executed"})

## 6 · One type, three semantics — the open policy question

> `- if: consensus_disagreement > 0.25` (§5.6, verbatim)

The rule does not say what the number is. Rather than choose for the
author, the specification fixes the metric's **type** — a fact-by-reader
answer matrix (answers in {agree, no, cantTell}) mapped to a scalar in
[0, 1] against the factored threshold — and exhibits **three
type-satisfying metrics with different semantics**, evaluated on the same
matrix. All three are correctly constructed. They do not agree on whether
the gate fires. **A correctly constructed policy is not the same as a
fit-for-purpose policy**: what is right for the job here is an open
algorithmic policy design question, likely to have no unique correct
answer but many viable answers with different implications on incentives —
and the right answer is not limited to these three.

- **dissent-fraction** counts every answer off the modal value, cantTell
  included: honest abstention is punished, so readers are pushed to guess.
  On the paper's own §5.5 example (three Cogs, two must agree) it FIRES
  the gate at 0.25 — contradicting the example's intent.
- **strict-quorum-undecodable** counts facts where no value reaches a
  quorum of two expressed answers; abstention blocks quorum, so it
  escalates more, spending expert-review capacity.
- **erasure-aware-undecodable** treats cantTell as an erasure, not an
  error: a lone expressed voice among abstentions decodes — abstention
  shifts power to whoever still answers.

The model holds the same openness structurally: the comparator oracle's
metric slot is typed by an abstract definition and deliberately unbound
(GAP-04, adjudicated open); the Track records which metric a run actually
used — recording is not endorsing.

In [ ]:
from typing import Callable, Sequence
DisagreementMetric = Callable[[Sequence[Sequence[str]]], float]
print("type signature: DisagreementMetric = (facts x readers answer matrix, '?' = cantTell) -> [0, 1]\n")

def dissent_fraction(matrix):
    dissents = total = 0
    for fact in matrix:
        expressed = [v for v in fact if v != "?"]
        modal = max(set(expressed), key=expressed.count) if expressed else None
        dissents += sum(1 for v in fact if v != modal)
        total += len(fact)
    return dissents / total

def strict_quorum_undecodable(matrix):
    undecodable = 0
    for fact in matrix:
        expressed = [v for v in fact if v != "?"]
        if not expressed or max(expressed.count(v) for v in set(expressed)) < 2:
            undecodable += 1
    return undecodable / len(matrix)

def erasure_aware_undecodable(matrix):
    undecodable = 0
    for fact in matrix:
        expressed = [v for v in fact if v != "?"]
        if not expressed or max(expressed.count(v) for v in set(expressed)) * 2 <= len(expressed):
            undecodable += 1
    return undecodable / len(matrix)

METRICS: dict[str, DisagreementMetric] = {
    "dissent-fraction": dissent_fraction,
    "strict-quorum-undecodable": strict_quorum_undecodable,
    "erasure-aware-undecodable": erasure_aware_undecodable,
}

matrix = [
    ["A", "A", "A"],
    ["A", "A", "A"],
    ["A", "A", "A"],
    ["A", "A", "B"],
    ["A", "A", "?"],
    ["A", "B", "?"],
    ["A", "B", "C"],
    ["A", "?", "?"],
]
threshold = float(re.search(r"consensusDisagreementThreshold : ScalarValues::Real = ([0-9.]+)", source).group(1))
print(f"same answer matrix for all three metrics ({len(matrix)} facts x 3 readers):")
for i, fact in enumerate(matrix, 1):
    print(f"  f{i}: " + " ".join(fact))
print(f"threshold (from the model, single point of definition): > {threshold}\n")
for name, metric in METRICS.items():
    value = metric(matrix)
    print(f"{name:26}: {value:.3f} -> gate fires: {value > threshold}")

## 7 · Wiring — the seams themselves are checked

`-validate -strict` accepts a seam whose ends do not match its interface
definition (probed against the pinned binary), so the structural wiring
rules live in code over the RDF conversion — the model holds the wiring,
the rules hold the wiring discipline. The same rule that passes the
committed assembly refuses the miswired counterexample, where an oracle's
reading port is wired into a summary input.

In [ ]:
import sys
sys.path.insert(0, "tests")
from test_wiring import check_seam_conformance, _converted
graph, _ = _converted(pathlib.Path(model))
mismatches = check_seam_conformance(graph)
seams = "confidenceSeam consensusSeam scannerSeam riskSeam obligationSeam clearanceSeam"
print(f"committed assembly ({len(seams.split())} seams): "
      + ("wiring rules: OK" if not mismatches else f"wiring rules: REFUSED {mismatches}"))
graph_bad, _ = _converted(pathlib.Path("counterexamples/miswired.sysml"))
bad = check_seam_conformance(graph_bad)
print("miswired counterexample: " + ("wiring rules: OK (?!)" if not bad else "wiring rules: REFUSED"))
for m in bad:
    print("  " + m)

## 8 · Lifecycle — the aggregate is not just an attribute

The assembly exhibits a lifecycle (running → stopped | completed) that runs
under the executor: a driver component delivers a signal across a connected
seam to the assembly's own port, and the trace shows the transition. Trace
output is the only honest post-run evidence. §5.2's stop is an actual state
the Op enters, not a label.

In [ ]:
def trace(context):
    r = subprocess.run(["toolchain/bin/sysml", model, "-trace", "-instantiate", context],
                       capture_output=True, text=True)
    lines = [l for l in r.stdout.splitlines() if "transition:" in l or "enter:" in l]
    return lines
for ctx in ("VendorFraudReview::ExerciseContexts::stoppedContext",
            "VendorFraudReview::ExerciseContexts::completedContext"):
    print(ctx.split("::")[-1])
    for line in trace(ctx)[:8]:
        print("  " + line)
    print()

## 9 · Conversion to RDF

The model converts to Turtle deterministically, and all three check levels
survive the conversion — so the same specification is queryable alongside
the vocabulary and the Track.

In [ ]:
c1 = subprocess.run(["toolchain/bin/sysml", model, "-convert", "ttl"], capture_output=True, text=True)
c2 = subprocess.run(["toolchain/bin/sysml", model, "-convert", "ttl"], capture_output=True, text=True)
mg = rdflib.Graph(); mg.parse(data=c1.stdout, format="turtle")
print(f"converted triples : {len(mg)}")
print(f"byte-stable       : {c1.stdout == c2.stdout}")
print(f"checks present    : {[rid for rid in ('GATE-01','GATE-02','GATE-03','GATE-04','WIRE-01','SYSTEM-01','SYSTEM-02','SYSTEM-03') if rid in c1.stdout]}")

## 10 · The Track

> "A Track is the durable record of an Op or Cog execution." (§5.3)

`track/run-001.trig` records the escalated run: three automatic Guard results
(passed, failed, and one honest *cantTell*), two fired Gates raising two
obligations, and one `earl:manual` approval by a named person that discharges
both obligations and generates the post-review case state (an action that
discharges an obligation must also mutate state). SHACL shapes derived from
the manifest's own `track.include` list check it — and refuse the
counterexample, where the obligations were raised and never discharged.
Retention of final_output is CONDITIONAL on the aggregate outcome (GAP-07
adjudication): an executed run must retain it, a noOp run must not — a
judgment that knowingly departs from the pinned draft's unconditional
include list, acknowledged in the shape messages themselves.

In [ ]:
from pyshacl import validate
def check(path):
    ds = rdflib.Dataset(default_union=True); ds.parse(path, format="trig")
    conforms, _, report = validate(ds, shacl_graph="shapes/track.shapes.ttl")
    return conforms, report
conforms, _ = check("track/run-001.trig")
print(f"run-001               conforms: {conforms}")
conforms, report = check("counterexamples/track-missing-approval.trig")
print(f"missing-approval      conforms: {conforms}")
print("\n".join(line for line in report.splitlines() if "Message" in line))

## 11 · Answering his own questions

§5.3 names the purposes a Track serves. Each is a query over the run graph,
not a reading exercise.

**Auditability** — "an organization can reconstruct why a decision was made":

In [ ]:
ds = rdflib.Dataset(default_union=True); ds.parse("track/run-001.trig", format="trig")
def show(rq):
    res = ds.query(pathlib.Path(rq).read_text())
    header = [str(v) for v in res.vars]
    print(" | ".join(header))
    for row in res:
        print(" | ".join(str(v) for v in row))
    return res
show("queries/auditability.rq");

**Governance** — "compliance teams can verify that required procedures were
followed". Rows are violations; empty means every obligation raised by a
fired Gate was discharged by a named human action. The same query catches
the counterexample:

In [ ]:
res = show("queries/governance.rq")
print(f"violations on run-001: {len(res)}")
cx = rdflib.Dataset(default_union=True); cx.parse("counterexamples/track-missing-approval.trig", format="trig")
cx_res = cx.query(pathlib.Path("queries/governance.rq").read_text())
print(f"violations on the counterexample: {len(cx_res)}")
for row in cx_res:
    print("  " + " | ".join(str(v) for v in row))

**Trust** — "users and customers can see that AI work was not merely
generated, but validated". The full outcome distribution, automatic and
manual, with *cantTell* visible rather than absorbed:

In [ ]:
show("queries/trust.rq");

**Interface** — where every value came from. The policy applies to
oracle-provided values; it is not their provider (GAP-05 ruling). For each
variable a gate evaluated: what service was called, what payload was sent,
what response code came back, and the response that carried the value. The
numerical precision lives inside the oracles or in documented threshold
rules; a reading nobody can source fails the shapes.

In [ ]:
res = ds.query(pathlib.Path("queries/interface.rq").read_text())
for row in res:
    d = row.asdict()
    print(f"{d['variable']} = {d['value']}")
    print(f"  service      : {d['service']}")
    print(f"  sent payload : {d['requestPayload']}")
    print(f"  responseCode : {d['responseCode']}")
    print(f"  response     : {d['responsePayload']}")

## 12 · Open questions

Where a literal parsing of the paper under-specifies what an executable
substrate requires, nothing was silently repaired: the choice in force is
marked provisional and logged. These entries are part of the demonstration.

In [ ]:
text = pathlib.Path("open-questions/computability-gaps.md").read_text()
for m in re.finditer(r"^## (GAP-\d+) — (.+?)$.*?\*\*Status:\*\* (\w+)", text, re.S | re.M):
    print(f"{m.group(1)}  [{m.group(3)}]  {m.group(2)}")

## 13 · Checks

One script runs everything on this page plus the test suite, and writes a
JSON report. The latest report:

In [ ]:
import json
report = pathlib.Path("checks/out/report.json")
if report.exists():
    print(json.dumps(json.loads(report.read_text()), indent=2))
else:
    print("no report yet — run: checks/run-checks.sh")